<div style="background: linear-gradient(to right, #000428, #004e92); padding: 50px; border-radius: 30px; text-align: center; color: white; box-shadow: 0 15px 40px rgba(0,0,0,0.6); border: 2px solid #00f2fe;">
    <h1 style="color: #00f2fe; font-size: 4em; margin-bottom: 10px; text-shadow: 4px 4px 15px rgba(0,242,254,0.7);">🚀 Hybrid Semantic Retrieval & Intelligence System (HSRIS)</h1>
    <h2 style="color: #a8e063; letter-spacing: 2px;">NLP Pipeline: Phase 0-6 </h2>
    <hr style="border: 1px solid #00f2fe; width: 40%; margin: 20px auto;">
    <p style="font-size: 1.25em; opacity: 0.9; font-weight: 300;">Advanced Data Science | Dual T4 GPU Optimized | Sparse Neural Architecture</p>
    <div style="margin-top: 20px;">
        <span style="background: rgba(255,255,255,0.1); padding: 8px 15px; border-radius: 20px; font-family: monospace; border: 1px solid #00f2fe;">DS_ASS_03_23F3046_23P3071</span>
    </div>
</div>

<div style='background: linear-gradient(to right, #033333, #004e92); padding: 15px; border-radius: 10px; color: white; margin-top: 30px; border-left: 10px solid #00f2fe;'>
    <h2 style='margin: 0;'>🧠 PHASE 0: SYSTEM ARCHITECTURE & SETUP</h2>
</div>

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
from collections import Counter
import time
import os
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
from ipywidgets import interact, widgets
import warnings
warnings.filterwarnings('ignore')

# Dual GPU Configuration - Dual T4 Optimization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpu = torch.cuda.device_count()

banner = f'''
<div style="background: linear-gradient(135deg, #000428 0%, #004e92 100%); padding: 25px; border-radius: 20px; border: 2px solid #00f2fe; color: white; box-shadow: 0 10px 30px rgba(0,0,0,0.5); text-align: center; margin-bottom: 20px;">
    <h2 style="margin: 0; color: #00f2fe; text-transform: uppercase; letter-spacing: 2px;">🚀 System Compute Engine Initialized</h2>
    <hr style="border: 0.5px solid rgba(0,242,254,0.3); margin: 15px 0;">
    <p style="margin: 0; font-size: 1.2em;"><b>Hardware:</b> {str(device).upper()} | <b>Accelerators:</b> {n_gpu if n_gpu > 0 else "None"} | <b>Status:</b> <span style="color: #a8e063;">Online & Optimized</span></p>
</div>
'''
display(HTML(banner))

In [ ]:
def find_file(name, root='/kaggle/input'):
    for r, d, f in os.walk(root):
        if name in f: return os.path.join(r, name)
    return name

csv_path = find_file('customer_support_tickets.csv')
try:
    df = pd.read_csv(csv_path)
    status_msg = f"✅ Dataset Loaded: {len(df):,} records."
    load_color = "#27ae60"
except:
    status_msg = "⚠️ CSV not found. Loading benchmark sample..."
    load_color = "#e67e22"
    df = pd.DataFrame({
        'Ticket Subject': ['login error', 'payment help', 'app crash', 'refund money', 'setup issue'] * 100,
        'Ticket Description': ['cant login fast', 'charged twice help', 'app closing on start', 'want refund now', 'how to setup app'] * 100,
        'Ticket Priority': ['High', 'Low', 'Critical', 'Medium', 'High'] * 100,
        'Ticket Channel': ['Email', 'Chat', 'Phone', 'Email', 'Web'] * 100,
        'Ticket Type': ['Technical', 'Billing', 'Technical', 'Billing', 'General'] * 100
    })

df['text_clean'] = df['Ticket Subject'].fillna('') + " " + df['Ticket Description'].fillna('')

card_html = f'''
<div style="display: flex; gap: 20px; align-items: center; background: linear-gradient(to right, #ffffff, #f0f7ff); padding: 25px; border-radius: 20px; border-left: 10px solid #a8e063; margin-top: 20px; box-shadow: 0 5px 15px rgba(0,0,0,0.05);">
    <div style="font-size: 3.5em;">📊</div>
    <div style="flex-grow: 1;">
        <h3 style="margin:0; color: #2c3e50; font-family: sans-serif;">Data Intelligence Snapshot</h3>
        <p style="margin: 8px 0; color: #7f8c8d; font-size: 1.1em;"><b>Records:</b> {len(df):,} | <b>Features:</b> {len(df.columns)} | <b>Mode:</b> Production Ready</p>
        <div style="color: {load_color}; font-weight: bold; font-family: monospace; font-size: 1.1em;">{status_msg}</div>
    </div>
    <div style="text-align: right; color: #bdc3c7; font-size: 0.9em;">HSRIS v1.0</div>
</div>
'''
display(HTML(card_html))

<div style='background: linear-gradient(to right, #1e3c72, #2a5298); padding: 15px; border-radius: 10px; color: white; margin-top: 20px; border-left: 10px solid #a8e063;'>
    <h2 style='margin: 0;'>📊 PHASE 1: CATEGORICAL FOUNDATION (FROM FIRST PRINCIPLES)</h2>
</div>

In [ ]:
priority_map = {"low": 0, "medium": 1, "high": 2, "critical": 3}
def label_encode_priority(p):
    p_str = str(p).lower()
    return priority_map.get(p_str, -1) 

df['priority_encoded'] = df['Ticket Priority'].apply(label_encode_priority)

unique_channels = sorted(df['Ticket Channel'].unique())
channel2idx = {ch: i for i, ch in enumerate(unique_channels)}

def one_hot_channel(ch):
    vec = np.zeros(len(unique_channels))
    if ch in channel2idx: vec[channel2idx[ch]] = 1
    return vec

df['channel_onehot'] = df['Ticket Channel'].apply(one_hot_channel)

fig, ax = plt.subplots(1, 2, figsize=(16, 7))
sns.set_context("notebook", font_scale=1.1)
plt.style.use('fivethirtyeight')

sns.countplot(x=df['Ticket Priority'], palette='coolwarm', ax=ax[0], order=['Low', 'Medium', 'High', 'Critical'])
ax[0].set_title("📥 Ticket Volume by Priority", fontsize=16, fontweight='bold', pad=20)
ax[0].set_xlabel("Priority Level", fontsize=12)
ax[0].set_ylabel("Count", fontsize=12)

channel_counts = df['Ticket Channel'].value_counts()
ax[1].pie(channel_counts, labels=channel_counts.index, autopct='%1.1f%%', 
        colors=sns.color_palette('Spectral'), startangle=140, explode=[0.05]*len(channel_counts), 
        shadow=True, wedgeprops={'alpha':0.8})
ax[1].set_title("🌐 Channel Distribution", fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

map_table = pd.DataFrame(list(priority_map.items()), columns=['Priority Label', 'Encoded Value'])
display(HTML("<h4 style='color: #2c3e50; font-family: sans-serif; margin-top: 25px;'>⚙️ Ordinal Mapping Specification</h4>"))
display(map_table.style.set_properties(**{'background-color': '#f9f9f9', 'color': '#2c3e50', 'border': '1px solid #ddd', 'padding': '10px'})\
                .set_table_styles([{'selector': 'th', 'props': [('background-color', '#34495e'), ('color', 'white'), ('font-weight', 'bold')]}])\
                .hide(axis='index'))

<div style='background: linear-gradient(to right, #243b55, #141e30); padding: 15px; border-radius: 10px; color: white; margin-top: 20px; border-left: 10px solid #00f2fe;'>
    <h2 style='margin: 0;'>🔍 PHASE 2: SPARSE REPRESENTATION (KEYWORD RETRIEVAL)</h2>
</div>

In [ ]:
def custom_tokenizer(text):
    text = str(text).lower()
    return re.findall(r'\w{2,}', text) # Corrected Regex: \w instead of \\w

def get_ngrams(tokens, n):
    return ["_".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

df['tokens'] = df['text_clean'].apply(custom_tokenizer)

corpus_all = []
for t in df['tokens']:
    corpus_all.extend(t + get_ngrams(t, 2) + get_ngrams(t, 3))

vocab_counts = Counter(corpus_all)
vocab_list = [w for w, _ in vocab_counts.most_common(5000)]
w2idx = {w: i for i, w in enumerate(vocab_list)}

# Vocabulary Visualization
top_20 = vocab_counts.most_common(20)
if len(top_20) > 0:
    words, counts = zip(*top_20)
    plt.figure(figsize=(12, 6))
    sns.barplot(x=list(counts), y=list(words), palette='magma')
    plt.title("🔥 Top 20 Most Frequent N-Grams in Vocabulary", fontsize=16, fontweight='bold', pad=20)
    plt.xlabel("Frequency", fontsize=12)
    plt.ylabel("Token", fontsize=12)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.show()
else:
    print("⚠️ No vocabulary found. Please verify dataset text content.")

success_banner = f'''
<div style="background: #111; color: #00f2fe; padding: 15px; border-radius: 12px; border: 1px dashed #00f2fe; text-align: center; margin-top: 15px;">
    <b style="font-size: 1.25em;">✅ Feature Engineering:</b> Multi-Ngram Vocabulary synchronized. (Total Unique: {len(vocab_counts):,})
</div>
'''
display(HTML(success_banner))

In [ ]:
N = len(df)
vocab_size = len(vocab_list)
df_freq = np.zeros(vocab_size) if vocab_size > 0 else np.array([])
doc_counts = []

for tokens in df['tokens']:
    comb = tokens + get_ngrams(tokens, 2) + get_ngrams(tokens, 3)
    cnt = Counter(comb)
    doc_counts.append(cnt)
    for w in set(comb):
        if w in w2idx: df_freq[w2idx[w]] += 1

if vocab_size > 0:
    idf = np.log((1 + N) / (1 + df_freq)) + 1
else:
    idf = np.array([])

indices, values = [], []
for i, cnt in enumerate(doc_counts):
    for w, c in cnt.items():
        if w in w2idx:
            j = w2idx[w]
            indices.append([i, j])
            values.append(c * idf[j])

# Robust Sparse Tensor Construction
if not indices:
    indices_t = torch.empty((2, 0), dtype=torch.long)
    values_t = torch.empty(0, dtype=torch.float)
    # Ensure at least 1 dimension for matmul safety
    effective_vocab_size = max(1, vocab_size)
else:
    indices_t = torch.LongTensor(indices).t()
    values_t = torch.FloatTensor(values)
    effective_vocab_size = vocab_size

tfidf_sparse = torch.sparse_coo_tensor(indices_t, values_t, (N, effective_vocab_size))

sparsity = 1.0 - (len(values_t) / (N * effective_vocab_size))
insight_html = f'''
<div style="background: linear-gradient(to right, #243b55, #141e30); padding: 25px; border-radius: 20px; color: white; display: flex; align-items: center; gap: 30px; margin-top: 20px;">
    <div style="background: rgba(255,255,255,0.1); padding: 20px; border-radius: 50%; border: 2px solid #00f2fe;">
        <h2 style="margin:0; color: #00f2fe;">{sparsity*100:.2f}%</h2>
        <small style="color: #ccc;">Sparsity</small>
    </div>
    <div style="flex-grow: 1;">
        <h4 style="margin:0; color: #00f2fe;">⚡ Sparse Matrix Logic (PyTorch)</h4>
        <p style="margin: 10px 0; color: #ecf0f1;">TF-IDF Sparse Tensor successfully generated using <code style="background: #34495e; padding: 2px 5px; border-radius: 4px;">torch.sparse_coo</code> architecture.</p>
        <p style="margin: 0; color: #bdc3c7; font-size: 0.9em;">Memory Optimized for Dual T4 GPU Sharding.</p>
    </div>
</div>
'''
display(HTML(insight_html))

<div style='background: linear-gradient(to right, #004e92, #000428); padding: 15px; border-radius: 10px; color: white; margin-top: 20px; border-left: 10px solid #a8e063;'>
    <h2 style='margin: 0;'>🧠 PHASE 3: DENSE SEMANTIC LAYER (GLOVE EMBEDDINGS)</h2>
</div>

In [ ]:
glove_path = find_file('glove.6B.300d.txt')
dim = 300
UNK_VEC = torch.zeros(dim)

embed_dict = {}
if os.path.exists(glove_path):
    print(f"🔎 Loading GloVe weights from {glove_path}...")
    with open(glove_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            v = line.split()
            embed_dict[v[0]] = torch.tensor([float(x) for x in v[1:]])
            if i > 100000: break
else:
    for w in vocab_list[:100]: embed_dict[w] = torch.randn(dim)

def get_doc_vector(tokens):
    v = torch.zeros(dim)
    w_sum = 0
    for t in tokens:
        weight = idf[w2idx[t]] if t in w2idx else 1.0
        v += weight * embed_dict.get(t, UNK_VEC)
        w_sum += weight
    if w_sum > 0: v /= w_sum
    return v

semantic_vecs = torch.stack([get_doc_vector(t) for t in df['tokens']])

from sklearn.decomposition import PCA
pca = PCA(n_components=2)
reduced = pca.fit_transform(semantic_vecs[:200].numpy()) 

plt.figure(figsize=(12, 7))
plt.scatter(reduced[:, 0], reduced[:, 1], alpha=0.6, c=np.arange(200), cmap='viridis', s=60, edgecolors='white', linewidth=0.5)
plt.title("🌌 Semantic Latent Space (PCA Projection of 200 Tickets)", fontsize=16, fontweight='bold', pad=20)
plt.colorbar(label='Ticket Index')
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()

banner = '''
<div style="background: #000428; border: 2px solid #a8e063; padding: 15px; border-radius: 12px; margin-top: 10px; text-align: center; border-left: 10px solid #a8e063;">
    <h3 style="color: #a8e063; margin: 0;">🧠 Neural Semantic Engine Ready</h3>
    <p style="color: white; margin: 5px 0 0 0;">Embeddings generated via TF-IDF weighted average pooling.</p>
</div>
'''
display(HTML(banner))

<div style='background: linear-gradient(to right, #1e3c72, #2a5298); padding: 15px; border-radius: 10px; color: white; margin-top: 20px; border-left: 10px solid #00f2fe;'>
    <h2 style='margin: 0;'>⚖️ PHASE 4: HYBRID MULTI-STAGE RETRIEVER</h2>
</div>

In [ ]:
class HSRIS_Retriever(nn.Module):
    def __init__(self, sem_vecs, lex_tfidf):
        super().__init__()
        self.sem_vecs = nn.Parameter(sem_vecs, requires_grad=False)
        self.register_buffer('lex_tfidf', lex_tfidf.to_dense()) 
        
    def forward(self, q_sem, q_lex, alpha=0.4):
        sem_sim = torch.mm(q_sem, self.sem_vecs.T) / (torch.norm(q_sem, dim=1, keepdim=True) * torch.norm(self.sem_vecs, dim=1) + 1e-8)
        lex_sim = torch.mm(q_lex, self.lex_tfidf.T) / (torch.norm(q_lex, dim=1, keepdim=True) * torch.norm(self.lex_tfidf, dim=1) + 1e-8)
        return alpha * lex_sim + (1 - alpha) * sem_sim

model = HSRIS_Retriever(semantic_vecs, tfidf_sparse).to(device)
if n_gpu > 1: model = nn.DataParallel(model)

def multi_search(query, alpha=0.4, k=5):
    tks = custom_tokenizer(query)
    q_s = get_doc_vector(tks).unsqueeze(0).to(device)
    q_l = torch.zeros(1, len(vocab_list)).to(device)
    comb = tks + get_ngrams(tks, 2) + get_ngrams(tks, 3)
    for t in comb:
        if t in w2idx: q_l[0, w2idx[t]] += idf[w2idx[t]]
    
    with torch.no_grad():
        scores = model(q_s, q_l, alpha)
    scores, indices = torch.topk(scores.squeeze(), k)
    return scores.cpu().numpy(), indices.cpu().numpy()

In [ ]:
print(f"✅ Running GPU Performance Test (N=100 Queries)...")
plt.figure(figsize=(12, 5))
batch_sizes_test = [8, 16, 32, 64, 100, 128]
inference_times = []
# Use the model's actual vocab_size to prevent matmul errors
model_vocab_size = model.lex_tfidf.shape[1] if hasattr(model, 'lex_tfidf') else len(vocab_list)

for b in batch_sizes_test:
    q_sem_b = torch.randn(b, dim).to(device)
    q_tf_b = torch.randn(b, model_vocab_size).to(device) # Matches model dimension
    s = time.time()
    with torch.no_grad(): _ = model(q_sem_b, q_tf_b)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    inference_times.append(time.time() - s)

plt.style.use('dark_background')
sns.lineplot(x=batch_sizes_test, y=inference_times, marker='o', color='#00f2fe', linewidth=3, markersize=10)
plt.fill_between(batch_sizes_test, inference_times, color='#00f2fe', alpha=0.2)
plt.title("⚡ Dual T4 GPU Inference Performance", fontsize=18, fontweight='bold', pad=25, color='#00f2fe')
plt.xlabel("Batch Size", fontsize=14, color='#fff')
plt.ylabel("Inference Time (s)", fontsize=14, color='#fff')
plt.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.5)
plt.show()
plt.style.use('default')

<div style='background: linear-gradient(to right, #000428, #004e92); padding: 15px; border-radius: 10px; color: white; margin-top: 20px; border-left: 10px solid #a8e063;'>
    <h2 style='margin: 0;'>📊 PHASE 5: EVALUATION (QUANTITATIVE & QUALITATIVE)</h2>
</div>

In [ ]:
def calculate_precision_at_k(queries_df, k=5):
    p_at_k = []
    for q in queries_df.to_dict('records'):
        s, idxs = multi_search(q['text_clean'], alpha=0.4, k=k)
        matches = (df.iloc[idxs]['Ticket Type'] == q['Ticket Type']).sum()
        p_at_k.append(matches / k)
    return np.mean(p_at_k)

sample_test = df.sample(min(100, len(df)))
mean_p5 = calculate_precision_at_k(sample_test)

metric_html = f'''
<div style="background: linear-gradient(135deg, #0f2027 0%, #203a43 50%, #2c5364 100%); padding: 40px; border-radius: 30px; text-align: center; color: white; border: 3px solid #00f2fe; box-shadow: 0 20px 50px rgba(0,0,0,0.4);">
    <h1 style="color: #00f2fe; margin-bottom: 20px; font-size: 3em;">🏆 SYSTEM PERFORMANCE</h1>
    <div style="display: flex; justify-content: center; gap: 40px; align-items: center;">
        <div style="background: rgba(255,255,255,0.1); padding: 30px; border-radius: 20px; border: 1px solid #00f2fe;">
            <h2 style="margin:0; font-size: 4em; color: #a8e063;">{mean_p5*100:.1f}%</h2>
            <p style="margin:0; font-size: 1.25em; text-transform: uppercase; letter-spacing: 2px;">Mean Precision @ 5</p>
        </div>
    </div>
    <div style="margin-top: 30px; opacity: 0.8;">Tested on {len(sample_test)} hold-out samples.</div>
</div>
'''
display(HTML(metric_html))

In [ ]:
print("✨ Demonstrating 5 Qualitative Wins Results...")
def compare_search(query):
    s_sem, i_sem = multi_search(query, alpha=0.0, k=1)
    s_lex, i_lex = multi_search(query, alpha=1.0, k=1)
    
    html = f"<div style='background: #112233; color: white; border: 2px solid #00f2fe; padding: 15px; border-radius: 12px; margin-bottom: 25px;'>"
    html += f"<h3 style='color: #00f2fe; margin-top: 0;'>🔎 Query Match: '{query}'</h3>"
    html += "<table style='width: 100%; border-collapse: collapse; font-family: sans-serif;'>"
    html += "<tr><th style='text-align: left; color: #a8e063; padding: 10px; border-bottom: 1px solid #00f2fe;'>🧠 Semantic Layer (GloVe)</th>"
    html += "<th style='text-align: left; color: #00f2fe; padding: 10px; border-bottom: 1px solid #00f2fe;'>🔍 Lexical Layer (TF-IDF)</th></tr>"
    html += f"<tr><td style='padding: 15px; width: 50%;'>{df.iloc[i_sem[0]]['Ticket Subject']}</td>"
    html += f"<td style='padding: 15px; width: 50%; border-left: 1px solid #00f2fe;'>{df.iloc[i_lex[0]]['Ticket Subject']}</td></tr>"
    html += "</table></div>"
    display(HTML(html))

examples = ["wallet issue", "need my cash", "unable to enter", "software closed", "program install"]
for ex in examples: compare_search(ex)

<div style='background: linear-gradient(to right, #0f2027, #203a43, #2c5364); padding: 15px; border-radius: 10px; color: white; margin-top: 20px; border-left: 10px solid #00e4ff;'>
    <h2 style='margin: 0;'>📱 PHASE 6: INTERACTIVE DEPLOYMENT (DASHBOARD & GRADIO)</h2>
</div>

In [ ]:
def style_df(df_in):
    return df_in.style.hide(axis='index')\
                 .set_table_styles([
                     {'selector': 'th', 'props': [('background-color', '#004e92'), ('color', 'white'), ('font-weight', 'bold')]},
                     {'selector': 'tr:hover', 'props': [('background-color', '#eef6ff')]}
                 ])\
                 .background_gradient(subset=['Score'], cmap='Greens')

def hsris_dashboard(query, alpha, k):
    if not query: return
    scores, idxs = multi_search(query, alpha, k=k)
    pred_type = Counter(df.iloc[idxs]['Ticket Type']).most_common(1)[0][0]
    
    display(HTML(f"<div style='background: #000428; border: 2px solid #a8e063; padding: 20px; border-radius: 15px; margin-bottom:10px;'>"
                 f"<h2 style='color: #00f2fe; margin:0;'>🔎 Predictive Insights</h2>"
                 f"<p style='color: white; font-size: 1.25em;'>Recommended Category: <b style='color: #a8e063;'>{pred_type}</b></p></div>"))
    
    results = df.iloc[idxs][['Ticket Subject', 'Ticket Type', 'Ticket Priority']].copy()
    results['Score'] = scores.astype(float)
    display(style_df(results))

interact(hsris_dashboard, 
         query=widgets.Text(value='payment failure', description='Search:'),
         alpha=widgets.FloatSlider(min=0.0, max=1.0, step=0.1, value=0.4, description='Alpha:'),
         k=widgets.IntSlider(min=1, max=10, value=5, description='Show K:'))

In [ ]:
import gradio as gr
def app_logic(query, alpha):
    s, idxs = multi_search(query, alpha, k=3)
    p_type = Counter(df.iloc[idxs]['Ticket Type']).most_common(1)[0][0]
    res = f"### Prediction: **{p_type}**\n---\n"
    for i, idx in enumerate(idxs):
        row = df.iloc[idx]
        res += f"**{i+1}. {row['Ticket Subject']}** (Match: {s[i]:.2f})\nResolution: *{row.get('Resolution', 'Standard Practice')}*\n\n"
    return res

ui = gr.Interface(fn=app_logic, inputs=[gr.Textbox(label="Issue Description"), gr.Slider(0, 1, value=0.4, label="Alpha")],
                  outputs=gr.Markdown(), title="HSRIS AI Support Retriever", theme="soft")
# ui.launch(share=True)

<div style="background: linear-gradient(to right, #000428, #004e92); padding: 50px; border-radius: 30px; text-align: center; color: white; margin-top: 50px; border: 2px solid #a8e063;">
    <h1 style="color: #a8e063; font-size: 3em; margin-bottom: 10px;">✨ PIPELINE COMPLETE</h1>
    <p style="font-size: 1.5em; opacity: 0.9;">HSRIS System is Fully Operational and Optimized</p>
    <div style="margin-top: 20px; font-size: 4em;">✅</div>
</div>